In [1]:
import json
# Sample JSON file
json_file = "../data-out/gems3/from_tmatch/2/DComp.exportGems3"

SDref = "dSDref"#"dSDref"
SDval = "dSDval"#"dSDval"
psi2020 = "2023HUM/THO" # "2024MIR"
tdb2020 = "2024MIR"
# change bellow to
                    #else:
                     #   dod[18]["val"][0] = "Miron:2025:rep:"
                     #   dod[19]["val"][0] = "PSINa25"
def read_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

def process_json(data):
    for item in data:
        print(f"Processing item with key: {item.get('key', [])}")
        for entry in item.get("dod", []):
            print(f"ID: {entry.get('id', 'N/A')}, Label: {entry.get('label', 'Unknown')}, Value: {entry.get('val', 'None')}")

def save_json(data, filename="formatted_references.json"):
    """Save formatted data to a JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [2]:
# Read and process JSON data
json_data = read_json(json_file+ ".json")
#process_json(json_data)
# Save formatted data to a JSON file
save_json(json_data, json_file+ ".json")

In [3]:
import pandas as pd
def load_references(file_path):
    """Load reference data from Excel file."""
    df = pd.read_excel(file_path, sheet_name="Sheet1")
    ref_dict = dict(zip(df["Ref_abb"].str.replace(" ", ""), df["Ref_GEMS"]))  # Remove spaces for comparison
    return ref_dict

In [4]:
# Load reference data
reference_file = "../data-out/gems3/from_tmatch/references.xlsx"
reference_data = load_references(reference_file)

In [5]:
def process_items_dSDref_dSDval(data, references):
    """Process item dSDref and update item dSDval based on reference mapping."""
    for item in data:
        dod = item.get("dod", [])
        dSDref = dod[SDref]
        dSDval = dod[SDval]
        if isinstance(dSDref, list) and isinstance(dSDval, list):
            updated_dSDval = []
            updated_dSDref = []
            for index, value in enumerate(dSDref):
                if ":" in value[0]:
                    # Case 1: If ':' exists, split and process both parts
                    first_part, second_part = value[0].split(":", 1)
                    second_part = second_part.replace(" ", "")  # Remove spaces
                    second_partb = second_part
                    second_part = references.get(second_part, second_part)  # Replace if match exists
                    updated_dSDval.append([first_part])
                    if second_part == second_partb:
                        updated_dSDref.append([f"{value}"])
                    else:
                        updated_dSDref.append([f"{second_part}"])
                else:
                    # Case 2: If no ':', check if value matches Ref_abb
                    #cleaned_value = value[0].replace(" ", "")  # Remove spaces for matching
                    #new_value = references.get(cleaned_value, value[0])  # Replace if match exists
                    updated_dSDval.append(dSDval[index])
                    #if value[0] == "TDB2020":
                    #    new_value = references.get(tdb2020, value[0])  # Replace if match exists
                    #    updated_dSDref.append([new_value])
                    #if value[0] == "PSI2020":
                    #    new_value = references.get(psi2020, value[0])  # Replace if match exists
                    #    updated_dSDref.append([new_value])
                    new_value = references.get(value[0], value[0])  # Replace if match exists
                    updated_dSDref.append([new_value])
            dod[SDval] = [entry for entry in updated_dSDval if entry and entry[0] != ""]  # 
            dod[SDref] = [entry for entry in updated_dSDref if entry and entry[0] != ""]   # 
        else:
            if isinstance(dSDref, list) :
                updated_dSDval = []
                updated_dSDref = []
                for index, value in enumerate(dSDref):
                    if ":" not in value[0]:
                        # Case 2: If no ':', check if value matches Ref_abb
                        cleaned_value = value[0].replace(" ", "")  # Remove spaces for matching
                        new_value = references.get(cleaned_value, value[0])  # Replace if match exists
                        updated_dSDval.append(value)
                        updated_dSDref.append([new_value])
                    else:
                        if "Vm_Ref:" in value[0]:
                            updated_dSDref.append(["Hummel_ea:2023:dat:"])  # Modify item 18
                            updated_dSDref.append(["Miron:2024:rep:"])  # Modify item 18
                            updated_dSDval.append(["TDB2020"]) 
                            updated_dSDval.append(["Vm_Ref"]) 
                        
                dod[SDval] = [entry for entry in updated_dSDval if entry and entry[0] != ""]  # 
                dod[SDref] = [entry for entry in updated_dSDref if entry and entry[0] != ""]   # 

        # Update DC_cnt based on size of dSDref
        dSDref = dod[SDref]
        if isinstance(dSDref, list):
            if "DC_cnt" in dod and isinstance(dod["DC_cnt"], list):
                if len(dod["DC_cnt"]) > 0 and len(dod["DC_cnt"][0]) > 2:
                    dod["DC_cnt"][0][2] = len(dSDref)

            
    return data

In [6]:
processed_data = process_items_dSDref_dSDval(json_data, reference_data)

In [7]:
#processed_data

def update_vm_ref_values(data):
    """Update dSDref with 'Miron:2024:rep' and dSDval with 'Vm_Ref' if 'Vm_Ref:' is present in item 18."""
    for item in data:
        dod = item.get("dod", [])
        dSDref = dod[SDref]
        dSDval = dod[SDval]

        if isinstance(dSDref, list) :#and isinstance(dSDval, list):
            updated_dSDref = []
            updated_dSDval = dSDval[:]  # Preserve original values from item 19

            for index, value in enumerate(dSDref):
                if "Vm_Ref:" in value[0]:
                    updated_dSval = []
                    updated_dSDref.append(["Hummel_ea:2023:dat:"])  # Modify item 18
                    updated_dSDref.append(["Miron:2024:rep:"])  # Modify item 18
                    updated_dSDval.append(["TDB2020"]) 
                    updated_dSDval.append(["Vm_Ref"]) 
                else:
                    updated_dSDref.append(value)  # Preserve item 18 values

            dod[SDval] = [entry for entry in updated_dSDval if entry and entry[0] != ""]  # 
            dod[SDref] = [entry for entry in updated_dSDref if entry and entry[0] != ""]   # 
            
        DC_cnt = dod["DC_cnt"] 
        if isinstance(dSDref, list) and DC_cnt is not None: 
             # Ensure DC_cnt has the expected structure 
            if len(DC_cnt) > 0 and len(DC_cnt[0]) > 5: 
                DC_cnt[0][5] += len(dSDref)
    return data

processed_data = update_vm_ref_values(processed_data)

In [8]:
#processed_data

In [9]:
# Save formatted data to a JSON file
save_json(processed_data, json_file+ "_formatted"+ ".json")